# 第七章：把机器学习变成一个可检验的金融研究

假设你每天收盘后都能得到某个资产的当日收益。你想知道：“最近涨过的资产，下一交易日还会继续涨吗？把最近五天的表现、波动一并交给模型，是否比只看昨天更有用？”这就是本章从头做到尾的问题。我们会先写出交易时点，构造一组确定性的合成行情，从历史均值走到线性回归和岭回归，最后把预测换算为扣成本后的仓位。每个模型都面对同一批样本；任何改进都必须能回到最初的问题解释。

这里的目标不是介绍尽可能多的算法名字。读完后，你应能亲手判断一个标签在训练时是否已经实现，解释正则化为什么能稳定相关特征，区分“预测误差下降”和“交易决策改善”，并写出新模型相对简单模型增加了什么证据。你还会认识分类、排序、树、主成分和神经网络在这条研究链中的具体位置，知道什么时候值得尝试，什么时候应该先停下来查数据。

默认实验只使用 NumPy 与标准库，无需网络、模型下载或行情账户。合成数据中故意存在弱可预测关系，后段关系还会改变；这让我们能观察研究方法的反应，不能据此推断真实市场存在同样规律。读者已熟悉统计推断，本章着重讲金融任务如何改变建模和检验。


## 1. 先把“预测明天”写清楚

我们研究单一资产的日内方向。设 \(r_t\) 是第 \(t\) 个交易日从开盘到收盘的简单收益，单位是小数：\(r_t=0.01\) 表示 1%。在 \(t\) 日收盘后，\(r_t\) 已经可见。模型使用截至此刻的特征 \(x_t\)，预测标签 \(y_t=r_{t+1}\)，即下一交易日的开盘到收盘收益。在 \(t+1\) 日开盘建立仓位，收盘全部平仓；隔夜不持仓，因此不把隔夜收益计入结果。这一约定也避开了“用收盘价算完信号，又按同一个收盘价成交”的矛盾。

**特征（feature）**是下决策时已知的输入，**标签（label）**是事后用来评分的目标。两者在表格中位于同一行，不代表它们同时可见。某行的标签若还没有实现，就不能进入这一时刻的训练集。**预测期限（horizon）**是从决策到标签结束的跨度，本例为一个交易日；**决策规则（decision rule）**是把预测转换成仓位的固定函数，稍后才定义。

一条可以复核的研究合同是：“用过去收益与波动预测下一日日内收益；以训练期均值预测为基线，以验证期均方误差选择岭回归强度；最终报告独立测试期的误差、净收益和持仓规模；每次进场和平仓各付 1 个基点。”**基点（basis point，bp）**等于 \(10^{-4}\)，所以两次各 1 bp 的费用，会让满仓一天的收益减少 0.0002，即 2 bp。

为什么用均值基线？如果模型只学到“这个样本整体偏涨”，一个常数预测就能做到。只有先打败这个**基线（baseline）**，才有理由说特征提供了信息。交易比较还需要一个**基准（benchmark）**：例如每天都做多。前者比较预测器，后者比较投资决策，两者用途不同。

在真实研究中，开盘价不是保证能成交的价格。我们把无冲击、可借券和完全成交作为教学假设；仓位为负时表示借入资产卖出后买回。借券费、开盘冲击、成交量约束都未建模，因而结果只是一份候选研究证据。


In [ ]:
import numpy as np

rng = np.random.default_rng(7)
n_days = 660
r = np.zeros(n_days)
for t in range(1, n_days):
    # 前段弱延续，后段弱反转；只用于观察模型面对关系变化的表现。
    persistence = 0.22 if t < 510 else -0.08
    r[t] = persistence * r[t - 1] + rng.normal(0.00015, 0.008)

decision_day = np.arange(20, n_days - 1)
X = np.array([[r[t], r[t - 4:t + 1].mean(),
               r[t - 19:t + 1].std(ddof=1)] for t in decision_day])
y = r[decision_day + 1]
feature_names = ["当日日内收益", "过去五日日内收益均值", "过去二十日波动"]
print("样本数、特征数:", X.shape)
print("第一行决策日、标签结束日:", decision_day[0], decision_day[0] + 1)
print("第一行特征 (bp):", np.round(X[0] * 10000, 2))


## 2. 从一行数据看见泄漏与样本依赖

代码中的第一行在第 20 日收盘决策。第一列只读第 20 日收益，第二列读第 16 至 20 日，第三列读第 1 至 20 日；标签读第 21 日。这里没有再对特征统一 `shift(1)`，因为我们已经在数组索引中写清了可见时间。机械地多移一日会改变研究问题；完全不管时间才会造成泄漏。是否需要 `shift` 取决于每列记录的含义和成交时点。

五日均值不是“过去五日买入持有收益”：我们的资产每天开盘进、收盘出，五个日内收益的平均值只作为状态特征。若研究五日连续持有，应该使用对应价格比或收益连乘，并把隔夜计入。这种单位与持有规则的区别，远比特征名是否漂亮更重要。

假如把标签改成未来五日收益，第 20 日的标签用第 21 至 25 日，第 21 日标签用第 22 至 26 日，它们共享四天。时间顺序切分仍不够：训练末尾的标签可能延伸到验证开始以后。**清除（purge）**就是删除这些训练标签尚未实现的行。**隔离期（embargo）**则是在特定评价协议中额外留出的间隔，用来降低邻近样本或事件依赖；它不是每个任务都必须机械套用的固定天数。

在本例中，验证第一天设为决策日 \(D\)。在 \(D\) 收盘拟合时，可用训练标签必须满足“标签结束日不晚于 \(D\)”。若规定模型在 \(D\) 开盘前冻结，条件就必须更严格。清除边界来自信息时钟，而不是一个库函数默认返回的行数。相同公司同一公告的多个版本还需要按事件分组，避免重复事件跨越训练与验证。




![相邻样本的 5 日前瞻标签互相重叠与 purge/embargo 位置](figures/ch07_label_overlap.png)

**图 7-1｜重叠标签与 purge/embargo。** 深蓝块为特征日，浅蓝块为未来 5 日标签窗口。跨越切分边界的训练标签需要 purge；额外 embargo 间隔应按标签跨度、信息延迟和评价协议确定。图示区间不代表所有任务都采用同一间隔。数据类型：标签窗口的概念示意。


## 3. 建立时间切分，把所有预处理锁在训练期

我们保留最初 360 行作训练，随后 130 行作验证，最后全部作测试。训练用于估计参数；验证用于在有限候选之间选择；测试回答“选完以后遇到新时期会怎样”。随机打乱会把后段反转关系混进前段，还会让邻近样本分散在两侧，因此不适合模拟本例的真实使用。

**标准化（standardization）**把每列变成相对其训练期均值、标准差的偏离：

\[
z_{tj}=(x_{tj}-\mu_j)/s_j.
\]

其中 \(x_{tj}\) 是决策日 \(t\) 的第 \(j\) 个特征，\(\mu_j\) 和 \(s_j\) 分别是训练行中的均值与标准差，\(z_{tj}\) 是无量纲输入。验证和测试必须沿用这两个训练参数。否则，连模型“一个标准差有多大”的定义都提前用了未来。

想象训练期波动为 1%，测试期为 3%。用全样本标准差缩放，会让模型在训练时就间接知道未来风险变大。这个影响即使不明显改变一个指标，也已经改变了可复现实验的含义。缺失值填补、极端值截尾、词表选择、主成分拟合都遵循同一规则：只在训练部分学习操作，再原样用于后续。

本例没有缺失。真实数据若缺一条报价，不应直接补成 0 收益，因为“确实未涨跌”与“根本没有观测”不同。可以记录缺失指示变量，或用当时已知的历史值填补，但先要分清停牌、尚未上市、数据供应中断与财报未披露。模型获得更高覆盖率，不一定获得了更可靠的信息。


In [ ]:
train = np.arange(0, 360)
valid = np.arange(360, 490)
test = np.arange(490, len(y))

# 在验证首个决策日收盘拟合，训练标签此时已全部实现。
assert (decision_day[train] + 1 <= decision_day[valid[0]]).all()

def fit_scaler(a):
    mean = a.mean(axis=0)
    scale = a.std(axis=0)
    return mean, np.where(scale > 1e-12, scale, 1.0)

mu, scale = fit_scaler(X[train])
Z = (X - mu) / scale
mean_prediction = np.full(len(valid), y[train].mean())
print("训练、验证、测试行数:", len(train), len(valid), len(test))
print("验证均值基线 MSE (bp²):",
      round(np.mean((y[valid] - mean_prediction) ** 2) * 1e8, 2))


## 4. 线性回归到底在学习什么

先用三个标准化特征预测收益：

\[
\widehat y_t=b+z_t^\top\beta.
\]

\(\widehat y_t\) 是对下一日日内收益的预测，\(b\) 是截距，\(z_t\) 是当前三个标准化特征组成的向量，\(\beta\) 是三个系数。某个系数为 0.001，表示其他输入不变时，该特征增加一个**训练期**标准差，预测收益增加 10 bp。它描述模型里的条件关联，不自动成为“这个特征造成收益变化”的因果结论。

本例第一列当日收益参与第二列五日均值，两列自然相关。**多重共线性（multicollinearity）**就是输入高度相关，导致许多不同系数组合产生相近预测。例如 \(z_2\) 几乎等于 \(z_1\)，系数 \((0.01,-0.009)\) 与 \((0.001,0)\) 的净效果差不多。样本稍变就可能令系数大幅跳动。这不是矩阵运算失败，而是数据无法清楚分辨两种解释。

**岭回归（Ridge regression）**给过大的系数加惩罚：

\[
\min_{b,\beta}\frac1n\sum_{t\in\mathcal T}(y_t-b-z_t^\top\beta)^2+\lambda\sum_j\beta_j^2.
\]

\(\mathcal T\) 是训练行集合，\(n\) 是训练行数，\(\lambda\geq0\) 控制收缩强度。我们不惩罚截距，因为不希望整体均值也被强行拉向零。由于损失写成平均值，下方正规方程使用 \(n\lambda\)；如果某个软件用误差平方和，参数数值不能直接照搬。

手算一个一维例子：特征已中心化，\(\sum z_t^2=4\)，\(\sum z_ty_t=0.008\)。普通最小二乘系数为 \(0.008/4=0.002\)。若 \(n\lambda=4\)，岭系数成为 \(0.008/(4+4)=0.001\)。它把对“每一标准差影响 20 bp”的激进估计缩成 10 bp，代价是若真实关系确实很强，预测会偏保守。

这就是偏差与方差的取舍。**偏差（bias）**指模型平均意义上的系统性误差，**方差（variance）**指更换训练样本后估计发生多大变化。更灵活的模型可以降低训练误差，也可能把噪声当规律；收缩允许一点偏差，换取更稳定的样本外预测。金融中的难点是弱关系、少量独立样本与时变机制同时存在。




![多项式阶数从 1 到 12 的训练与测试 MSE 曲线](figures/ch07_bias_variance.png)

**图 7-2｜训练与验证误差。** 复杂度增加时训练误差下降，验证误差用于选择阶数；最终测试集应保留到模型确定后评价。本例高阶验证误差略有上升，不必形成明显 U 形。数据类型：固定 seed 多项式回归样本；真实金融任务还须按时间切分。


In [ ]:
def fit_ridge(z, target, lam):
    z_mean = z.mean(axis=0)
    y_mean = target.mean()
    centered_z = z - z_mean
    centered_y = target - y_mean
    gram = centered_z.T @ centered_z + len(target) * lam * np.eye(z.shape[1])
    beta = np.linalg.lstsq(gram, centered_z.T @ centered_y, rcond=None)[0]
    intercept = y_mean - z_mean @ beta
    return intercept, beta

lambdas = [0.0, 0.01, 0.1, 1.0, 10.0]
validation_rows = []
for lam in lambdas:
    intercept, beta = fit_ridge(Z[train], y[train], lam)
    prediction = intercept + Z[valid] @ beta
    mse = np.mean((y[valid] - prediction) ** 2)
    validation_rows.append((mse, lam))
    print("lambda =", lam, "验证 MSE (bp²) =", round(mse * 1e8, 2))
best_lambda = min(validation_rows)[1]
print("按验证误差冻结的 lambda:", best_lambda)


## 5. 损失函数、模型家族与研究问题一一对应

上一步使用**均方误差（mean squared error，MSE）**，即 \(n^{-1}\sum_t(y_t-\widehat y_t)^2\)。这里 \(n\) 是评分样本数，误差平方与原收益单位不同，所以代码以 bp² 显示。平方损失重罚大错，通常适合学习条件均值；若少数异常报价支配结果，可以检查数据后再考虑绝对误差或 Huber 损失。Huber 在小误差区间保留平方形状，大误差区间改用线性增长，它改变估计目标，不能作为删除不喜欢交易亏损的借口。

如果把本例改成“下一日是否上涨”，标签变为 \(d_t=1\{y_t>0\}\)。**逻辑回归（logistic regression）**用 \(p_t=(1+e^{-\eta_t})^{-1}\) 输出上涨概率，其中 \(\eta_t=b+z_t^\top\beta\) 是线性打分。训练常用对数损失，惩罚自信而错误的概率。它属于**广义线性模型（generalized linear model，GLM）**：通过连接函数把条件均值映射到线性预测子；连续收益常用恒等连接，二元事件用逻辑连接，事件次数可用对数连接的泊松模型。标签分布和业务问题先确定，模型名随后确定。

本例保持回归，是因为上涨 1 bp 与上涨 100 bp 的交易价值不同。假设十次中九次上涨 1 bp，一次下跌 30 bp，永远预测上涨准确率达到 90%，但每次固定做多的总毛收益为 \(9-30=-21\) bp。分类准确率遗漏了收益幅度，更遗漏了成本。

若扩展到每天许多资产，还可以只学习资产的相对次序，称为**排序学习（ranking）**。研究者常报告排序信息系数（rank IC）：同日预测排名与未来收益排名的相关程度。它回答“高分资产是否相对更好”，并不回答多空组合能赚多少。我们这里只有一个资产，后面算的是测试期不同日期间的相关系数，不冒充横截面 IC。

复杂模型何时有意义？**决策树（decision tree）**通过“波动是否超过阈值”之类的条件分段；它能表达当前收益只在低波动期有效。**随机森林（random forest）**平均多棵扰动后的树，降低单树不稳定性。**梯度提升（gradient boosting）**逐步拟合已有模型的剩余误差，需要严格控制树深和迭代次数。**神经网络（neural network）**用多层非线性变换学习表示，适合充分样本和复杂输入，但三个数值特征不自动要求深层结构。

这些算法属于**监督学习（supervised learning）**，都有标签。**无监督学习（unsupervised learning）**则用于发现输入结构，例如主成分分析（PCA）寻找训练期方差最大的正交方向。本例把两列相关收益压缩为主成分可能稳定估计，但最大方差方向未必最能预测下一日；PCA 的均值、缩放和载荷也必须仅在训练期拟合。**强化学习（reinforcement learning）**优化一串动作的累积奖励，适合研究库存与执行，但要建立可信环境、成本和风险约束。本例一步仓位问题先用监督模型即可，增加动作反馈之前应先能解释它比固定决策规则改善了什么。


## 6. 冻结选择，然后只问一次测试期发生了什么

选出 \(\lambda\) 后，我们允许用训练与验证合并的数据重估一次缩放与系数，再进入测试期。验证数据此时已成为拟合样本，不能继续被称为样本外证据；测试仍保持未使用。最终同时报告均值预测、只用当日收益的岭模型和三特征岭模型。后两个模型使用同一个 \(\lambda\)，这样能看清增加特征的作用；若要各自选参数，必须在验证期各自完成，而不能从测试结果倒推。

**扩展窗口（expanding window）**保留全部过去样本，**滚动窗口（rolling window）**只保留最近固定长度，**向前滚动检验（walk-forward）**是在每个新时点只用此前已实现标签重训并评分。我们这里采取一次冻结，便于看清流程。真实研究要比较重训方式，必须把窗口长度和重训频率也当作选择参数，在验证阶段确定。测试里看到后段反转后再缩短窗口，会重新引入选择偏差。

单次划分不能证明稳定性。独立测试期含有后段机制变化，恰好能提醒我们：合成关系曾经存在，不意味着对所有未来一直有效。若三特征模型没有明显优于均值，正确做法是陈述证据不足、检查置信范围与时期贡献，不是把测试期改名为验证期继续挑到满意。


In [ ]:
fit_rows = np.concatenate([train, valid])
mu_final, scale_final = fit_scaler(X[fit_rows])
Z_final = (X - mu_final) / scale_final
b1, beta1 = fit_ridge(Z_final[fit_rows, :1], y[fit_rows], best_lambda)
b3, beta3 = fit_ridge(Z_final[fit_rows], y[fit_rows], best_lambda)
predictions = {
    "训练均值": np.full(len(test), y[fit_rows].mean()),
    "仅当日收益": b1 + Z_final[test, :1] @ beta1,
    "三特征 Ridge": b3 + Z_final[test] @ beta3,
}
for model_name, pred in predictions.items():
    mse = np.mean((y[test] - pred) ** 2) * 1e8
    corr = np.corrcoef(y[test], pred)[0, 1] if pred.std() > 1e-12 else np.nan
    print(model_name, "测试 MSE (bp²):", round(mse, 2), "跨日期相关:", round(corr, 3))
print("三特征系数 (bp/训练标准差):", np.round(beta3 * 10000, 2))


### 把实际输出读成一个研究结论

在固定种子的这次实验里，验证期选择的 \(\lambda\) 为 1。测试期均值预测的 MSE 约为 4899 bp²，三特征模型约为 4968 bp²；三特征跨日期预测相关约 -0.044。也就是说，验证中看起来合理的模型，没有在后来的新时期打败最简单的均值预测。这个结果与合成机制后段变弱并反转相容，但单凭一段负相关，仍不能在真实数据里断言已经发生结构突变。

同时，三特征模型的净收益约为 0.51 bp/日，而只用当日收益约为 1.99 bp/日；双倍成本使三特征版本转负。这不是“多两列输入使算法变差”的普遍定律，而是这份样本在冻结协议下没有给出新增特征值得保留的证据。最诚实的后续是暂时保留简单版本作为研究基线，把新增特征放回待验证假设。

注意均值预测不是零预测。它在训练加验证段估出一个负均值，因此映射后持续偏空。测试期恰好也有负平均收益，这能解释为何它的净收益优于每日做多。这个优势可能主要是方向暴露，不能把它写成模型发现了逐日择时。看到一个好数字后回到仓位和收益分布，是区分统计模型与交易叙事的重要习惯。

## 7. 从预测到交易：成本如何改变问题

固定仓位函数为

\[
w_t=\operatorname{clip}(\widehat y_t/a,-1,1),\qquad
q_t=w_ty_t-2c|w_t|.
\]

\(w_t\) 是在决策日 \(t\) 生成、下一日开盘执行的资产权重，允许在 -1 至 1 间变化；\(a=0.002\) 是事先固定的收益尺度，即预测达到 20 bp 就满仓；\(c=0.0001\) 是单边成本；\(q_t\) 是下一日日内净收益。因为每天平仓，进出交易量合计为 \(2|w_t|\)，费用必须两边都收。若改为连续持有，交易量应是相邻目标仓位之差，不能沿用这条日内公式。

用一笔交易手算：预测 +10 bp，仓位为 0.5；下一日实际 -40 bp，毛收益为 -20 bp，进出总成本 1 bp，净收益 -21 bp。如果预测从 +10 bp 改善到 +6 bp，平方误差有所下降，仓位也降为 0.3，这笔亏损减小。反过来，模型在许多接近零的日子只改善了极小误差，却产生更大的来回交易，净结果可能变差。这就是为什么不能只看 MSE。

这里的 \(a\) 没有在测试期调到最高收益，所有模型共享它。这个选择将模型比较与仓位优化分开，方便归因；它也未必最优。若引入**波动率目标（volatility targeting）**，可让仓位按历史波动倒数缩放，但估计窗口、最大仓位、停牌与极端波动处理都成为新的规则，必须在新测试之前冻结。

对于分类模型，概率还要经过**校准（calibration）**。若把所有输出 0.8 的样本放在一起，实际上涨比例应该接近 80%。但即使概率校准，上涨和下跌的幅度仍决定期望收益：80% 赚 1 bp、20% 亏 10 bp 的期望仍为 -1.2 bp。Platt 方法用逻辑函数重映射原分数，等距回归用单调函数映射；两者都要在独立的历史校准段学习，不能用测试标签修正测试概率。




![原始模型概率与校准后概率相对实现频率的校准曲线](figures/ch07_calibration.png)

**图 7-3｜概率必须经得起频率检验。** 对角线表示预测概率等于实现频率。原始概率经过幂变换而系统性偏低，示意映射后更接近对角线；图中没有训练独立的校准器，也不能据此报告样本外校准表现。数据类型：固定 seed 的合成概率与二元标签。


In [ ]:
per_side_cost = 0.0001
position_scale = 0.002

def trade_summary(pred, actual, cost=per_side_cost):
    weight = np.clip(pred / position_scale, -1.0, 1.0)
    volume = 2 * np.abs(weight)  # 每天都从空仓开始，并在收盘回到空仓。
    gross = weight * actual
    net = gross - cost * volume
    return {
        "gross_bp_day": gross.mean() * 10000,
        "net_bp_day": net.mean() * 10000,
        "volume_day": volume.mean(),
        "net_series": net,
    }

trade_results = {}
for model_name, pred in predictions.items():
    result = trade_summary(pred, y[test])
    doubled = trade_summary(pred, y[test], 2 * per_side_cost)
    trade_results[model_name] = result
    print(model_name, "毛/净 bp每天:", round(result["gross_bp_day"], 3),
          round(result["net_bp_day"], 3), "双倍成本净 bp每天:",
          round(doubled["net_bp_day"], 3), "平均双边交易量:",
          round(result["volume_day"], 3))
print("每日满仓做多净 bp每天:", round((y[test] - 2 * per_side_cost).mean() * 10000, 3))


## 8. 新特征是否增加了独立信息

三特征模型优于一特征模型，可能因为五日均值包含互补信息，也可能只是拟合偶然。最直接的研究方法是**消融（ablation）**：删去一组输入，从相同训练期重新拟合，在同一保留期比较。上面的“仅当日收益”就是一个消融版本。完整实验还应把参数选择预算保持一致，并记录多次尝试，否则在许多删减组合中挑出最好者也会过拟合。

**置换重要性（permutation importance）**是在评分数据上扰动某列，观察误差增加多少。若扰动当日收益导致误差上升，说明模型的预测依赖它；这不证明当日收益导致未来涨跌。相关特征能互相替代，逐列置换可能低估各自作用；时间序列逐行乱排又破坏了序列结构。金融任务更适合比较预先定义的特征组消融、按连续区块扰动，必要时用块重抽样评估差异的不确定性。

**特征重要性与因果效应必须分开。** 本例波动系数为负，不意味着“人为提高波动会降低明天收益”。它可能只代理了某段市场状态；合成机制中我们根本没有给它这样的因果路径。线性系数、SHAP 值和偏依赖图都描述特定模型如何使用输入，其中 SHAP 将输出分配给输入，偏依赖图改变某个输入并平均其他输入；它们都受相关结构和训练分布限制。


### 预测不确定性如何约束仓位

点预测只给一个数，例如“明天 +8 bp”，却没有说明这个估计有多可靠。至少要区分两种不确定性：即使真实条件均值已知，下一日仍有价格噪声；训练样本有限、参数变化又使条件均值本身估不准。增加数据主要帮助后者，不会让市场的随机波动消失。本例模型预测通常只有几个 bp，而日内波动约几十 bp，单笔方向错误十分常见，不能用一次盈亏检验整个模型。

一种朴素诊断是按照连续历史窗口重复拟合，观察同一当前输入得到的预测是否稳定。若有的窗口预测 +10 bp、有的预测 -10 bp，就应承认估计脆弱，而不是把平均值接近零称为“模型一致”。多模型平均可以降低部分估计方差，但若所有模型共享错误标签或同一数据泄漏，集成不会纠正共同错误。

**保形预测（conformal prediction）**常用历史残差的分位数给点预测加一个区间。若校准段绝对残差的高分位数是 \(q\)，可形成 \([\widehat y_t-q,\widehat y_t+q]\)。这里 \(q\) 是在独立历史校准段学习的误差尺度，不能用未来残差估计。标准覆盖保证依赖样本可交换等条件，金融时序与漂移往往不满足；滚动校准与分状态检查只能帮助诊断，不能宣称无条件保证。即使区间经常覆盖实际收益，它也未必窄到足以支持交易。

对本例最合理的动作是先约束最大仓位并报告稳定性，而不是从很小的预测直接放大杠杆。如果要新增“只有区间完全在成本阈值之外才交易”的规则，需要在验证阶段确定，再用新保留期比较拒绝率、损失和交易结果。**元标签（meta-label）**也是类似的第二层任务：已有方向信号后，学习何时参与或放大。它增加一个新模型与选择层，不能在原测试表现上挑参与日后继续沿用原测试结论。

### 一个可执行的后续实验应有明确分工

如果怀疑五日均值只有在低波动期有效，可以预先加入“当日收益乘低波动标记”这一交互项。交互项让一个输入的效应取决于另一个输入；低波动阈值只能从训练期确定。随后先检查新增特征覆盖多少样本，再按同一验证预算比较岭模型与浅树。若树改善的是训练而非验证，说明灵活度尚未获得证据支持；若两者都只在一个时期改善，应先收集独立时期。

回归目标若改为未来相对市场收益，也会改变结论含义。原始收益包含方向风险，减去一个当时约定的市场基准后，模型更接近预测相对表现；但风险系数若使用全样本估计，又会引入未来信息。不要把“换标签”“中性化”“改仓位”同时做完再把总改善归给算法，每次改变都应能对应独立的研究问题。




![带 bootstrap 误差棒的 permutation importance](figures/ch07_importance_stability.png)

**图 7-4｜重要性与不确定度。** 在固定 seed 的合成 iid 回归中，对训练集做 30 次 bootstrap，每次在独立留出集逐特征置换，记录 MSE 增量。柱高是均值，误差棒为 ±1 标准差；它们不是置信区间，也不能支持真实交易信号或因果效应的结论。数据类型：可复现的合成回归与置换计算。


把问题扩展到多资产后，还要区分“模型单独表现好”与“对现有组合有增量”。假如三特征模型几乎复制原有趋势信号，独立收益很好，但加入组合只增加重复交易。应该同时比较单独结果、与基线合并后的边际收益、边际风险和额外换手。**截面中性化（cross-sectional neutralization）**是在同一天对许多资产的分数回归行业、规模等已知暴露，再用残差作信号。它能减少共同风险，也可能移除有效信息。我们这里只有一个资产，无法假装通过横截面回归去掉行业；更合理的是报告每日多空偏向、与做多基准的相关性，以及收益是否集中在同一方向。

任何变换都应回到本章的三列输入检查：何时可用、多久平滑、是否截尾、覆盖多少日子、如何转仓、增加多少成本。所谓表达式审计，就是把这些具体选择列明。模型名字不会自动替你解决延迟、平滑衰减、单个权重截断和风险暴露。复杂文本模型如果降低覆盖或增加处理时间，也必须在相同决策时点下重做增量比较。


## 9. 当关系变化时，监控什么

合成序列在后段把收益延续系数从 0.22 改成 -0.08。历史拟合仍可能给出正系数，这时模型并没有计算错，而是旧关系不再适用。**概念漂移（concept drift）**指输入与标签之间的关系变化；**特征漂移（feature drift）**指输入分布变化；**标签漂移（label drift）**指目标分布变化。另有执行层变化，例如真实成本增加，即使预测关系不变，也会使净收益失效。

这几种现象要分别诊断。波动变大不代表延续系数变号，几天亏损也不代表模型永久失效。部署前可以约定：每日检查字段缺失和特征是否越出训练范围；标签成熟后按固定窗口比较预测残差；每月按冻结规则重训；如果成本或数据质量超过阈值，先缩仓或停止发新单。阈值必须结合误报成本、样本依赖与处理能力设置，而不能看完一段曲线再挑最漂亮的报警线。




![同一特征在 2018–2023 年的分布山脊图与 KS 统计量](figures/ch07_feature_drift.png)

**图 7-5｜特征漂移的可见形态。** 各年份的合成特征分布不同，2021 年后均值右移且波动扩大；2019 与 2023 的双样本 KS 统计量约为 0.50。分布漂移提示重新检查标准化与模型假设，不等于所有阈值或预测能力都已失效。数据类型：固定 seed 合成特征。


In [ ]:
full_pred = predictions["三特征 Ridge"]
for block_name, block in [("测试前半", slice(0, len(test)//2)),
                         ("测试后半", slice(len(test)//2, None))]:
    result = trade_summary(full_pred[block], y[test][block])
    print(block_name, "净 bp每天:", round(result["net_bp_day"], 3))
print("测试特征均值偏移 (训练标准差单位):",
      np.round(Z_final[test].mean(axis=0), 2))


## 10. 用真实数据延伸时，保留哪条主线

本地若有经过许可的日线开盘价和收盘价，可以用 \(C_t/O_t-1\) 替代合成的 \(r_t\)，继续使用同样的三列特征、同样的训练/验证/测试流程。\(O_t\) 与 \(C_t\) 分别表示该交易日开盘和收盘价格，必须统一币种与公司行动处理。开盘缺失、停牌、退市与复权方式需要先检查；若数据只有收盘价，就不能把收盘到收盘收益塞进本例并仍声称是无隔夜交易。

真实资产通常不能按指数点位买卖。如果只有指数数据，应明确它只是代理；ETF、期货等可交易工具有跟踪误差、费用、保证金和换月问题。默认实验不读取本地文件，因此任何人都能运行；真实延伸应作为另一个有完整数据合同的实验。项目 B 的模型层也采用均值和 NumPy 岭回归，其价值就在于可以检查这条主线，而不是隐藏在大型框架里。

研究下一步应来自失败诊断。例如三特征模型在高波动段失败，可以先固定分组，检查误差和成本是否共同上升；若只在少数大跌日失败，应排查方向暴露与尾部风险；若始终打不过均值，可以接受“当前输入未提供足够样本外信息”。增加一棵树或一个神经网络只有在它对应一个可检验的非线性假设时才值得消耗新的验证预算。

## 11. 常见误解：回到本例逐一拆开

“标准化不使用标签，所以全样本算也没关系。”标准化仍学习未来输入分布；本例训练时不知道未来波动范围。“把每行 `shift(1)` 就没有泄漏。”只有明确信息时间与成交时点才知道该移动什么，标签清除与重复事件也不会被统一移动修好。

“岭系数小，说明这个特征没用。”系数还依赖单位、缩放与相关输入；当日收益和五日均值可能分摊解释。“测试 MSE 最低就是最好的策略。”仓位映射、收益幅度、方向偏向与成本都可能改变排序。“看到新数据就不断改参数是适应市场。”未经预设地追逐最近表现，会把运行期变成新的反复选择集；先写重训协议和停止规则，才能谈稳定使用。


## 12. 手工练习

### 练习一：给一行数据写时间合同

第 40 日收盘后，特征为当日收益和五日均值，标签改为第 41 至 45 日的累计日内复合收益。第 43 日收盘开始验证，并在此刻拟合模型。训练最多能保留哪一天的决策行？若不清除，会发生什么？

<details><summary>参考答案</summary>

第 \(t\) 行标签到 \(t+5\) 日收盘才全部实现，要求 \(t+5\leq43\)，所以最多保留第 38 日决策行。第 39 至 42 日虽在验证决策前，其标签仍包含第 44 日以后的收益，训练当时不可得。这里允许恰在第 43 日收盘实现的标签；若模型必须在收盘前冻结，就应进一步退一日。问题关键是标签实现时间，不能只比较决策行日期。

</details>

### 练习二：预测变好是否值得交易

某模型连续三天预测分别为 +4、-8、+20 bp，实际分别为 +10、+10、-5 bp。按本章 20 bp 满仓的线性映射，单边成本 1 bp，计算三日净收益。若设“预测绝对值不超过 5 bp 则不交易”，结论怎样？

<details><summary>参考答案</summary>

原仓位为 0.2、-0.4、1。毛收益分别为 +2、-4、-5 bp，双边费用为 0.4、0.8、2 bp，净收益为 +1.6、-4.8、-7 bp，合计 -10.2 bp。阈值规则去掉第一天，合计变为 -11.8 bp。阈值不是天然改进；本例删掉了盈利交易。若看完这三天再选阈值，这三天也不能继续充当独立测试。

</details>

### 练习三：解释相关特征

当日收益与五日均值相关，三特征岭模型中第一列系数从 12 bp 变为 4 bp，第二列从 0 变为 9 bp。能否说当日收益信息减少了三分之二？你会怎样设计一个更可信的比较？

<details><summary>参考答案</summary>

不能。系数受到缩放、相关性和惩罚共同影响，信息可能被重新分配。先保持样本、预处理、成本和调参预算一致，分别拟合只用当日收益、只用五日均值和两者合用的模型，再比较保留期预测误差、净收益和时期稳定性。相关模型的差异还需要用连续时间区块评估不确定性，不能把逐日差异当独立重复。

</details>

## 13. 自测与思考

1. 为什么本章无需统一再对三列特征 `shift(1)`？什么时候必须改变？
2. 训练损失由平方和改成平均值后，为什么相同数值的正则化参数不再等价？
3. 分类概率完全校准，是否足以决定做多？
4. 主成分解释了最多方差，为什么不保证有预测能力？
5. 若测试前半盈利、后半亏损，下一步应该怎样区分关系变化和成本变化？

<details><summary>参考答案</summary>

第一，行索引已经明确表示收盘可见输入、次日开盘执行；若改为当日开盘执行，必须只读此前已知信息。第二，平均损失比平方和小了 \(n\) 倍，惩罚的相对权重随之改变。第三，还需要上涨/下跌幅度、成本、风险和持仓约束。第四，PCA 学习输入结构而不使用预测标签。第五，按预先固定时间块分别比较特征分布、残差、方向暴露、毛收益、成交量与实际单位成本；一次亏损无法单独识别原因，需要新的时间段或数据支持。

</details>

## 14. 本章小结

同一组三列历史特征，依次连接了标签时间、预处理、均值基线、岭回归、验证选择、独立测试、仓位与双边成本。每新增一种算法，都应回到这条链解释它改变了哪个环节、增加了多少选择自由度，以及是否在相同规则下贡献了独立信息。

一个完整结论可以是：“在冻结划分和日内交易规则下，这个模型的预测误差表现如何，净结果是否覆盖费用，哪些时期失败；当前证据支持继续检查哪项机制，尚不支持直接部署。”能够清楚说出这句话，比背熟模型名单更接近真正的金融机器学习研究。
